# TT/MPS基礎 10 — Eckart–Young–Mirsky 定理と単一 Bond Truncation の最適性

## 今回の位置づけ

前回の Notebook 09 では、3階 TT/MPS

$$
X
=
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}
$$

の第2中心コアを左展開し、

$$
A
=
G_2^{[C]\langle L\rangle}
\in
\mathbb{R}^{(r_1n_2)\times r_2}
$$

に truncated SVD を適用しました。

教材用の具体例は、

$$
A\in\mathbb{R}^{6\times3},
\qquad
\rho=\operatorname{rank}(A)=3,
\qquad
k=2,
$$

$$
(\sigma_1,\sigma_2,\sigma_3)
=
(6,\;2,\;0.25)
$$

でした。

Notebook 09 では、

$$
\|X-\widetilde X\|_F
=
0.25
$$

および、

$$
\|X-\widetilde X\|_F^2
=
0.0625
=
\sigma_3^2
$$

を数値で確認しました。

ただし、そこまでで分かったのは、

> truncated SVD を使うと、この誤差になった

ということです。

今回はさらに、

> **rank を $k$ 以下にするという制約のもとでは、これより小さい Frobenius 誤差を持つ近似は存在しない**

ことを Eckart–Young–Mirsky（EYM）定理で確認します。

### 今回やること

1. EYM 定理が何を主張するか確認する
2. 候補行列 $B$ は rank 制約を満たせば任意であることを確認する
3. Notebook 09 の「小さい特異値を捨てる」に最適性を与える理由を理解する
4. 一般の Frobenius 版 EYM の証明を追う
5. 証明の核となる von Neumann のトレース不等式を追う
6. mixed-canonical TT/MPS の単一 bond truncation へ翻訳する
7. 次の3実験を自分で実装して確認する
   - 実験A：任意の rank-$2$ 候補と EYM 最適解を比較
   - 実験B：von Neumann のトレース不等式を数値確認
   - 実験C：mixed-canonical 環境が局所誤差を保存することを確認

### 今回はまだ扱わないもの

- tolerance による rank 決定
- relative error による rank 決定
- TT rounding の sweep
- 複数 bond にまたがる全体誤差評価
- entanglement entropy
- DMRG / ALS
- TT-matrix / MPO

また、von Neumann のトレース不等式の証明では **Ky Fan の最大値原理を既知の定理として引用**します。Ky Fan の原理自体の完全証明は、この Notebook の範囲外です。


## 1. Eckart–Young–Mirsky 定理の主張

一般に、

$$
A\in\mathbb{R}^{m\times n},
\qquad
\rho=\operatorname{rank}(A),
\qquad
0\le k<\rho
$$

とします。

SVD を、

$$
A
=
U\Sigma V^T
=
\sum_{\alpha=1}^{\rho}
\sigma_\alpha
u_\alpha v_\alpha^T,
$$

$$
\sigma_1
\ge
\sigma_2
\ge
\cdots
\ge
\sigma_\rho
>
0
$$

とします。

先頭の $k$ 成分だけを残した truncated SVD を、

$$
A_k
=
\sum_{\alpha=1}^{k}
\sigma_\alpha
u_\alpha v_\alpha^T
=
U_k\Sigma_kV_k^T
$$

と定義します。

EYM 定理の Frobenius ノルム版は、

$$
\boxed{
A_k
\in
\operatorname*{argmin}_{\operatorname{rank}(B)\le k}
\|A-B\|_F
}
$$

と主張します。

つまり、

$$
\boxed{
\|A-A_k\|_F
\le
\|A-B\|_F
\qquad
\text{for every }B\text{ with }\operatorname{rank}(B)\le k
}
$$

です。

さらに最小二乗誤差は、

$$
\boxed{
\|A-A_k\|_F^2
=
\sum_{\alpha=k+1}^{\rho}
\sigma_\alpha^2
}
$$

なので、

$$
\boxed{
\min_{\operatorname{rank}(B)\le k}
\|A-B\|_F
=
\sqrt{
\sum_{\alpha=k+1}^{\rho}
\sigma_\alpha^2
}
}
$$

です。

Notebook 09 の例では、

$$
(\sigma_1,\sigma_2,\sigma_3)
=
(6,2,0.25),
\qquad
k=2,
$$

なので、

$$
\boxed{
\min_{\operatorname{rank}(B)\le2}
\|A-B\|_F
=
0.25
}
$$

となります。


## 2. 候補 $B$ は「rank 条件を満たせば何でもよい」

ここで重要なのは、

$$
B\in\mathbb{R}^{m\times n},
\qquad
\operatorname{rank}(B)\le k
$$

以外に、$B$ の作り方へ制約がないことです。

Notebook 09 の例なら、

$$
B\in\mathbb{R}^{6\times3},
\qquad
\operatorname{rank}(B)\le2
$$

を満たせば候補です。

例えば、

- $A$ の特異ベクトルと無関係な rank-$2$ 行列
- 特異ベクトルを回転した rank-$2$ 行列
- $A$ の第1・第3 SVD 成分を残した行列
- rank-$1$ 行列
- 零行列

もすべて候補に含まれます。

したがって EYM 定理は、

> **SVD 成分の中からどれを残すかだけを比較している定理ではない**

ことが重要です。

任意の rank-$k$ 以下の行列という、連続的で非常に広い候補集合の中で truncated SVD が最適です。

rank-$k$ 以下の候補は例えば、

$$
B
=
PQ^T,
$$

$$
P\in\mathbb{R}^{m\times k},
\qquad
Q\in\mathbb{R}^{n\times k}
$$

のように作れます。


## 3. Notebook 09 の例での直感

Notebook 09 の中心行列は、

$$
A
=
6u_1v_1^T
+
2u_2v_2^T
+
0.25u_3v_3^T.
$$

SVD の rank-$1$ 成分

$$
u_\alpha v_\alpha^T
$$

は Frobenius 内積で互いに正規直交します。

$$
\left\langle
u_\alpha v_\alpha^T,
u_\beta v_\beta^T
\right\rangle_F
=
\delta_{\alpha\beta}.
$$

もし SVD の3成分から2成分だけを選ぶなら、

$$
\begin{array}{c|c}
\text{残す特異値} & \text{Frobenius 誤差}\\
\hline
(6,2) & 0.25\\
(6,0.25) & 2\\
(2,0.25) & 6
\end{array}
$$

なので、最小の $0.25$ を捨てるのが最もよいことは直感的に分かります。

ただし EYM 定理が保証するのは、これより強い主張です。

$$
\boxed{
\text{どんな rank-2 行列 }B\text{ を作っても、誤差 }0.25\text{ 未満にはできない}
}
$$

つまり、方向を回転した別の rank-$2$ 近似を考えても、truncated SVD を上回れません。


## 4. なぜ Notebook 09 の $\rho=3,\ k=2$ は特別に分かりやすいか

Notebook 09 では、

$$
\rho-k
=
3-2
=
1
$$

でした。

つまり捨てる非ゼロ特異値が1個だけです。

この場合、

$$
\sum_{\alpha=k+1}^{\rho}
\sigma_\alpha^2
=
\sigma_3^2
$$

なので、Frobenius 誤差は、

$$
\|A-A_2\|_F
=
\sigma_3
=
0.25
$$

です。

rank-$2$ 行列 $B\in\mathbb{R}^{6\times3}$ は、入力3次元のうち少なくとも1方向を kernel に持ちます。

したがって1本の「失われる方向」を使う次元論だけでも、

$$
\|A-B\|_F
\ge
\sigma_3
$$

まで到達できます。

しかし一般に、

$$
\rho-k\ge2
$$

なら、捨てる特異値は複数あります。

目標は、

$$
\|A-B\|_F^2
\ge
\sigma_{k+1}^2
+
\sigma_{k+2}^2
+
\cdots
+
\sigma_\rho^2
$$

です。

1本のベクトルだけを使う kernel の議論では、通常、

$$
\sigma_{k+1}^2
$$

という「捨てた中で最大の1方向」までしか同時に捉えられません。

そこで一般の Frobenius 版には、複数の特異値を同時に扱う道具として **von Neumann のトレース不等式**を使います。


## 5. 証明で使う基本量：Frobenius 内積

行列 $P,Q$ の Frobenius 内積を、

$$
\langle P,Q\rangle_F
=
\operatorname{tr}(P^TQ)
$$

とします。

Frobenius ノルムは、

$$
\|P\|_F^2
=
\langle P,P\rangle_F
=
\operatorname{tr}(P^TP)
$$

です。

したがって、

$$
\begin{aligned}
\|A-B\|_F^2
&=
\langle A-B,A-B\rangle_F\\
&=
\|A\|_F^2
-2\langle A,B\rangle_F
+\|B\|_F^2.
\end{aligned}
$$

実数行列では、

$$
\langle A,B\rangle_F
=
\operatorname{tr}(A^TB).
$$

EYM の一般証明では、この交差項

$$
\operatorname{tr}(A^TB)
$$

を特異値だけで上から押さえることが重要になります。


## 6. von Neumann のトレース不等式

$A,B$ の特異値を、

$$
\sigma_1(A)\ge\sigma_2(A)\ge\cdots\ge0,
$$

$$
\sigma_1(B)\ge\sigma_2(B)\ge\cdots\ge0
$$

と並べます。

von Neumann のトレース不等式は、

$$
\boxed{
\left|
\operatorname{tr}(A^TB)
\right|
\le
\sum_i
\sigma_i(A)\sigma_i(B)
}
$$

です。

EYM の証明では、特に、

$$
\operatorname{tr}(A^TB)
\le
\sum_i
\sigma_i(A)\sigma_i(B)
$$

を使います。

直感的には、

> $A$ と $B$ の Frobenius 内積は、両者の「大きい特異方向どうし」を最大限そろえた場合を超えられない

という不等式です。

これにより、$B$ の具体的な特異ベクトル方向を知らなくても、**特異値だけで任意の候補 $B$ を一括して評価**できます。


## 7. von Neumann 不等式の証明で引用する Ky Fan の最大値原理

この Notebook では、Ky Fan の最大値原理を既知の定理として引用します。

任意の行列

$$
M\in\mathbb{R}^{m\times n}
$$

と、正規直交列を持つ、

$$
X\in\mathbb{R}^{m\times k},
\qquad
Y\in\mathbb{R}^{n\times k},
$$

$$
X^TX
=
I_k,
\qquad
Y^TY
=
I_k
$$

に対して、

$$
\boxed{
\operatorname{tr}(X^TMY)
\le
\sum_{i=1}^{k}\sigma_i(M)
}
$$

が成り立ちます。

さらに $X,Y$ を $M$ の上位 $k$ 個の左右特異ベクトルに選ぶと、上界を達成します。

ここでは Ky Fan の原理そのものは証明せず、von Neumann 不等式を導くための土台として使います。


## 8. Abel 和分で $B$ を部分和へ分解する

説明を簡単にするため、まず正方行列の場合を考えます。

$B$ の SVD を、

$$
B
=
\sum_{j=1}^{n}
\sigma_j(B)p_jq_j^T
$$

とします。

部分和を、

$$
S_k
=
\sum_{j=1}^{k}
p_jq_j^T,
\qquad
S_0=0,
$$

さらに、

$$
\sigma_{n+1}(B)
=
0
$$

と定義します。

このとき Abel 和分により、

$$
\boxed{
B
=
\sum_{k=1}^{n}
\left(
\sigma_k(B)-\sigma_{k+1}(B)
\right)
S_k
}
$$

と書けます。

実際、第 $j$ rank-$1$ 成分の係数を集めると、

$$
\sum_{k=j}^{n}
\left(
\sigma_k(B)-\sigma_{k+1}(B)
\right)
=
\sigma_j(B)
$$

という望遠鏡和になります。


## 9. von Neumann のトレース不等式を導く

Abel 和分を、

$$
\operatorname{tr}(A^TB)
$$

へ代入します。

$$
\operatorname{tr}(A^TB)
=
\sum_{k=1}^{n}
\left(
\sigma_k(B)-\sigma_{k+1}(B)
\right)
\operatorname{tr}(A^TS_k).
$$

ここで、

$$
S_k
=
P_kQ_k^T,
$$

$$
P_k
=
[p_1,\ldots,p_k],
\qquad
Q_k
=
[q_1,\ldots,q_k]
$$

と書けば、

$$
\operatorname{tr}(A^TS_k)
=
\operatorname{tr}(P_k^TAQ_k).
$$

Ky Fan の最大値原理より、

$$
\operatorname{tr}(P_k^TAQ_k)
\le
\sum_{i=1}^{k}
\sigma_i(A).
$$

また特異値は降順なので、

$$
\sigma_k(B)-\sigma_{k+1}(B)
\ge0.
$$

したがって、

$$
\operatorname{tr}(A^TB)
\le
\sum_{k=1}^{n}
\left(
\sigma_k(B)-\sigma_{k+1}(B)
\right)
\sum_{i=1}^{k}\sigma_i(A).
$$

二重和の順序を入れ替えると、

$$
\begin{aligned}
&\sum_{k=1}^{n}
\left(
\sigma_k(B)-\sigma_{k+1}(B)
\right)
\sum_{i=1}^{k}\sigma_i(A)
\\
&=
\sum_{i=1}^{n}
\sigma_i(A)
\sum_{k=i}^{n}
\left(
\sigma_k(B)-\sigma_{k+1}(B)
\right).
\end{aligned}
$$

内側は再び望遠鏡和なので、

$$
\sum_{k=i}^{n}
\left(
\sigma_k(B)-\sigma_{k+1}(B)
\right)
=
\sigma_i(B).
$$

よって、

$$
\boxed{
\operatorname{tr}(A^TB)
\le
\sum_{i=1}^{n}
\sigma_i(A)\sigma_i(B)
}
$$

が得られます。

これが EYM の一般 Frobenius 版で使う von Neumann のトレース不等式です。


## 10. 一般の Frobenius 版 EYM を証明する

任意の、

$$
B\in\mathbb{R}^{m\times n},
\qquad
\operatorname{rank}(B)\le k
$$

を考えます。

まず、

$$
\|A-B\|_F^2
=
\|A\|_F^2
+
\|B\|_F^2
-
2\operatorname{tr}(A^TB).
$$

Frobenius ノルム二乗は特異値の二乗和なので、

$$
\|A\|_F^2
=
\sum_i\sigma_i(A)^2,
$$

$$
\|B\|_F^2
=
\sum_i\sigma_i(B)^2.
$$

von Neumann の不等式

$$
\operatorname{tr}(A^TB)
\le
\sum_i\sigma_i(A)\sigma_i(B)
$$

に $-2$ を掛けると不等号の向きが反転し、

$$
-2\operatorname{tr}(A^TB)
\ge
-2\sum_i\sigma_i(A)\sigma_i(B).
$$

したがって、

$$
\begin{aligned}
\|A-B\|_F^2
&\ge
\sum_i\sigma_i(A)^2
+
\sum_i\sigma_i(B)^2
-
2\sum_i\sigma_i(A)\sigma_i(B)
\\
&=
\sum_i
\left(
\sigma_i(A)-\sigma_i(B)
\right)^2.
\end{aligned}
$$

よって、

$$
\boxed{
\|A-B\|_F^2
\ge
\sum_i
\left(
\sigma_i(A)-\sigma_i(B)
\right)^2
}
$$

を得ます。


## 11. Rank 制約を入れる

いま、

$$
\operatorname{rank}(B)\le k
$$

なので、$B$ の特異値は、

$$
\sigma_i(B)=0
\qquad
(i>k)
$$

です。

したがって、

$$
\begin{aligned}
\sum_i
\left(
\sigma_i(A)-\sigma_i(B)
\right)^2
&=
\sum_{i=1}^{k}
\left(
\sigma_i(A)-\sigma_i(B)
\right)^2
\\
&\quad+
\sum_{i=k+1}^{n}
\sigma_i(A)^2.
\end{aligned}
$$

第1項は二乗和なので、

$$
\sum_{i=1}^{k}
\left(
\sigma_i(A)-\sigma_i(B)
\right)^2
\ge0.
$$

よって、

$$
\boxed{
\|A-B\|_F^2
\ge
\sum_{\alpha=k+1}^{\rho}
\sigma_\alpha(A)^2
}
$$

です。

この下界は、rank-$k$ 以下の **任意の** $B$ に対して成立します。


## 12. Truncated SVD が下界を達成する

$$
B=A_k=U_k\Sigma_kV_k^T
$$

を選びます。

$A_k$ の特異値は、

$$
\sigma_i(A_k)
=
\begin{cases}
\sigma_i(A), & i\le k,\\
0, & i>k.
\end{cases}
$$

なので、

$$
\sum_{i=1}^{k}
\left(
\sigma_i(A)-\sigma_i(A_k)
\right)^2
=
0.
$$

また $A$ と $A_k$ は同じ特異ベクトル方向を共有するため、von Neumann の不等式でも等号になります。

したがって、

$$
\boxed{
\|A-A_k\|_F^2
=
\sum_{\alpha=k+1}^{\rho}
\sigma_\alpha^2
}
$$

です。

任意の $B$ に対する下界と、$A_k$ が実際に達成する値が一致したので、

$$
\boxed{
\min_{\operatorname{rank}(B)\le k}
\|A-B\|_F^2
=
\|A-A_k\|_F^2
=
\sum_{\alpha=k+1}^{\rho}\sigma_\alpha^2
}
$$

が証明されました。

証明の依存関係は、

$$
\boxed{
\text{Ky Fan（引用）}
\rightarrow
\text{von Neumann}
\rightarrow
\text{特異値差の下界}
\rightarrow
\text{EYM}
}
$$

です。


## 13. 今回の数値検証用セットアップ

Notebook 09 と同じ、

$$
n_1=4,\quad
n_2=3,\quad
n_3=5,\quad
r_1=2,\quad
r_2=3
$$

を使います。

左右の環境コアは canonical にし、中心行列の特異値を、

$$
(6,\;2,\;0.25)
$$

へ固定します。

今回のコード演習では、この同じ $A$ と TT/MPS を、

- 実験A
- 実験B
- 実験C

で共通して使います。


In [1]:
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)


def reconstruct_tt3(
    G1: torch.Tensor,
    G2: torch.Tensor,
    G3: torch.Tensor,
) -> torch.Tensor:
    """3個のTTコアから3階テンソルを再構成する。"""
    return torch.einsum("aib,bjc,ckd->ijk", G1, G2, G3)


# shape
n1, n2, n3 = 4, 3, 5
r1, r2 = 2, 3
k = 2

# --- 左 canonical 環境 ---
G1_seed = torch.randn(1, n1, r1)
Q1, _ = torch.linalg.qr(G1_seed.squeeze(0), mode="reduced")
G1_left = Q1.unsqueeze(0)

# --- 右 canonical 環境 ---
G3_seed = torch.randn(r2, n3, 1)
Q3, _ = torch.linalg.qr(G3_seed.squeeze(-1).T, mode="reduced")
G3_right = Q3.T.unsqueeze(-1)

# --- 特異値を制御した中心行列 A ---
A_random = torch.randn(r1 * n2, r2)
U_seed, _ = torch.linalg.qr(A_random, mode="reduced")

V_seed, _ = torch.linalg.qr(
    torch.randn(r2, r2),
    mode="reduced",
)

S_target = torch.tensor([6.0, 2.0, 0.25])

A = U_seed @ torch.diag(S_target) @ V_seed.T
G2_center = A.reshape(r1, n2, r2)

X = reconstruct_tt3(
    G1_left,
    G2_center,
    G3_right,
)

print("A shape        :", tuple(A.shape))
print("G1_left shape  :", tuple(G1_left.shape))
print("G2_center shape:", tuple(G2_center.shape))
print("G3_right shape :", tuple(G3_right.shape))
print("target singular values:", S_target)


A shape        : (6, 3)
G1_left shape  : (1, 4, 2)
G2_center shape: (2, 3, 3)
G3_right shape : (3, 5, 1)
target singular values: tensor([6.0000, 2.0000, 0.2500])


## 14. 演習A — 任意の Rank-$2$ 候補は EYM 下界を破れるか

### 目的

EYM の主張、

$$
\|A-B\|_F
\ge
\|A-A_2\|_F
$$

を、複数のランダム rank-$2$ 候補 $B$ に対して数値確認します。

ランダム候補は、

$$
B_j
=
P_jQ_j^T,
$$

$$
P_j\in\mathbb{R}^{6\times2},
\qquad
Q_j\in\mathbb{R}^{3\times2}
$$

として作れます。

この形なら、

$$
\operatorname{rank}(B_j)\le2
$$

です。

### TODO

1. `A` に reduced SVD を適用する
2. truncated SVD から `A_k` を作る
3. 
   $$
   \|A-A_k\|_F
   $$
   を計算する
4. ランダムな rank-$2$ 候補を複数作る
5. 各候補について
   $$
   \|A-B_j\|_F
   $$
   を計算する
6. 最良のランダム候補の誤差を求める
7. float64 の小さい許容誤差 $\varepsilon$ を考慮して、
   $$
   \min_j\|A-B_j\|_F
   \ge
   \|A-A_k\|_F-\varepsilon
   $$
   を確認する

### 考えること

- ランダム探索は EYM の証明になっているか
- ランダム候補が最適値 $0.25$ に近づかない場合、それは定理と矛盾するか
- EYM の強さは「試した候補」ではなく「すべての rank-$2$ 候補」に対する主張であること


In [4]:
# TODO A:
# 任意の rank-2 候補 B と truncated SVD A_k の誤差を比較してください。
#
from nn_compression.compression.svd import truncated_svd, matrix_max_rank

# 1. reduced SVD
rank = matrix_max_rank(A)
k = 2
print("A shape        :", tuple(A.shape))
print("rank (exact max):", rank)
print("k (keep)        :", k)
print("discarded       :", rank - k)
assert 1 <= k <= rank
U, S, Vh = truncated_svd(A, rank)
U_k, S_k, Vh_k = U[:, :k], S[:k], Vh[:k, :]
#
# 2. A_k を構成
A_k = U_k @ torch.diag(S_k) @ Vh_k

# 3. EYM 最適誤差を計算
eym_error = torch.linalg.vector_norm(A - A_k).item()
eym_error_from_S = S[k:].square().sum().sqrt().item()
print("EYM error ||A-A_k||_F =", eym_error)
print("from discarded S     =", eym_error_from_S)
#
# 4. P @ Q.T でランダム rank-2 候補を複数作る
#    B = P @ Q.T なら必ず rank(B) <= k。
#    最適とは限らない「条件を満たす候補」をたくさん集めて EYM 下界を数値確認する。
n_trials = 50
m, n = A.shape  # (6, 3)
Bs = []
for _ in range(n_trials):
    P = torch.randn(m, k)  # (6, 2)
    Q = torch.randn(n, k)  # (3, 2)
    Bs.append(P @ Q.T)  # B: (6, 3), rank(B) <= k
#
# 5. 各候補について ||A - B_j||_F を計算
#    あとで最良値と ||A - A_k||_F を比較する
random_errors = torch.tensor(
    [torch.linalg.vector_norm(A - B).item() for B in Bs]
)
#
# 6. 最良のランダム候補と A_k を比較
best_random_error = random_errors.min().item()
print("best random ||A-B||_F =", best_random_error)
print("mean random ||A-B||_F =", random_errors.mean().item())
#
# 7. 許容誤差を入れて不等式を確認
eps = 1e-12
print("best_random >= eym - eps ?", best_random_error >= eym_error - eps)

A shape        : (6, 3)
rank (exact max): 3
k (keep)        : 2
discarded       : 1
EYM error ||A-A_k||_F = 0.25000000000000006
from discarded S     = 0.2500000000000006
best random ||A-B||_F = 6.08920060366182
mean random ||A-B||_F = 8.093946171967817
best_random >= eym - eps ? True


## 15. 演習B — von Neumann のトレース不等式を数値確認する

### 目的

一般証明の核になった、

$$
\left|
\langle A,B\rangle_F
\right|
\le
\sum_i
\sigma_i(A)\sigma_i(B)
$$

を、ランダムな $B$ で確認します。

Frobenius 内積は、

$$
\langle A,B\rangle_F
=
\operatorname{tr}(A^TB)
=
\sum_{i,j}A_{ij}B_{ij}
$$

です。

### TODO

1. ランダムな候補 `B` を1つ、または複数作る
2. 左辺
   $$
   |\langle A,B\rangle_F|
   $$
   を計算する
3. `A` と `B` の特異値を求める
4. 右辺
   $$
   \sum_i\sigma_i(A)\sigma_i(B)
   $$
   を計算する
5. 左辺が右辺以下であることを確認する
6. 可能なら、複数の `B` に対して繰り返す

### 考えること

- なぜ `B` の特異ベクトル方向を直接使わなくても比較できるのか
- EYM の証明で、この不等式が `B` の全候補をまとめて扱える理由
- `A` と `B` が同じ特異ベクトル方向を共有するとき、等号に近づく理由


In [5]:
# TODO B:
# von Neumann のトレース不等式を数値確認してください。
#
# 1. 候補 B を作る
#    A と同じ shape のランダム行列（複数）
n_trials = 20
m, n = A.shape
Bs = [torch.randn(m, n) for _ in range(n_trials)]

# 2. |<A, B>_F| を計算
# 3. A, B の特異値を求める
# 4. sum_i sigma_i(A) sigma_i(B) を計算
# 5. 不等式を確認
S_A = torch.linalg.svdvals(A)

ok_count = 0
gaps = []
for B in Bs:
    # 左辺: |<A,B>_F| = |sum_{ij} A_ij B_ij|
    lhs = torch.abs((A * B).sum()).item()

    # 右辺: sum_i σ_i(A) σ_i(B)
    S_B = torch.linalg.svdvals(B)
    # 特異値の本数が違う場合に備え、短い方に合わせる
    r = min(S_A.numel(), S_B.numel())
    rhs = (S_A[:r] * S_B[:r]).sum().item()

    gaps.append(rhs - lhs)
    if lhs <= rhs + 1e-12:
        ok_count += 1

print("von Neumann holds for", ok_count, "/", n_trials, "trials")
print("min (rhs - lhs) =", min(gaps))
print("mean (rhs - lhs) =", sum(gaps) / len(gaps))

# 参考: 1本だけ詳細表示
B0 = Bs[0]
lhs0 = torch.abs((A * B0).sum()).item()
S_B0 = torch.linalg.svdvals(B0)
r0 = min(S_A.numel(), S_B0.numel())
rhs0 = (S_A[:r0] * S_B0[:r0]).sum().item()
print("example |<A,B>_F| =", lhs0)
print("example sum σA σB =", rhs0)
print("example gap        =", rhs0 - lhs0)

von Neumann holds for 20 / 20 trials
min (rhs - lhs) = 12.810197235713535
mean (rhs - lhs) = 19.778032524496826
example |<A,B>_F| = 0.9440644364745796
example sum σA σB = 25.399403223600945
example gap        = 24.455338787126365


## 16. TT/MPS への翻訳：局所行列の誤差が全体誤差になる理由

mixed-canonical form

$$
X
=
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}
$$

を考えます。

第2中心コアの左展開を、

$$
A
=
G_2^{[C]\langle L\rangle}
$$

とします。

任意の同じ shape の行列、

$$
B
\in
\mathbb{R}^{(r_1n_2)\times r_2}
$$

を中心コアへ戻したものを、

$$
G_2^{(B)}
=
\operatorname{reshape}(B)
$$

とします。

対応するテンソルを、

$$
X^{(B)}
=
G_1^{[L]}
G_2^{(B)}
G_3^{[R]}
$$

とします。

差分は、

$$
\Delta G_2
=
G_2^{[C]}-G_2^{(B)}
$$

で、

$$
\Delta X(i_1,i_2,i_3)
=
\sum_{\alpha_1,\alpha_2}
G_1^{[L]}(1,i_1,\alpha_1)
\Delta G_2(\alpha_1,i_2,\alpha_2)
G_3^{[R]}(\alpha_2,i_3,1).
$$

mixed-canonical 条件は、

$$
\sum_{i_1}
G_1^{[L]}(1,i_1,\alpha_1)
G_1^{[L]}(1,i_1,\beta_1)
=
\delta_{\alpha_1\beta_1},
$$

$$
\sum_{i_3}
G_3^{[R]}(\alpha_2,i_3,1)
G_3^{[R]}(\beta_2,i_3,1)
=
\delta_{\alpha_2\beta_2}.
$$

です。


## 17. Isometry による誤差保存を添字で導く

差分テンソルのノルム二乗を展開します。

$$
\begin{aligned}
\|\Delta X\|_F^2
&=
\sum_{i_1,i_2,i_3}
\Delta X(i_1,i_2,i_3)^2
\\
&=
\sum_{i_1,i_2,i_3}
\sum_{\alpha_1,\alpha_2}
\sum_{\beta_1,\beta_2}
G_1^{[L]}(1,i_1,\alpha_1)
G_1^{[L]}(1,i_1,\beta_1)
\\
&\qquad\qquad\cdot
\Delta G_2(\alpha_1,i_2,\alpha_2)
\Delta G_2(\beta_1,i_2,\beta_2)
\\
&\qquad\qquad\cdot
G_3^{[R]}(\alpha_2,i_3,1)
G_3^{[R]}(\beta_2,i_3,1).
\end{aligned}
$$

$i_1$ の和で、

$$
\delta_{\alpha_1\beta_1}
$$

が出ます。

$i_3$ の和で、

$$
\delta_{\alpha_2\beta_2}
$$

が出ます。

したがって交差項が消えて、

$$
\begin{aligned}
\|\Delta X\|_F^2
&=
\sum_{i_2,\alpha_1,\alpha_2}
\Delta G_2(\alpha_1,i_2,\alpha_2)^2
\\
&=
\|\Delta G_2\|_F^2.
\end{aligned}
$$

reshape は要素を並べ替えるだけなので、

$$
\|\Delta G_2\|_F
=
\|A-B\|_F.
$$

よって、

$$
\boxed{
\|X-X^{(B)}\|_F
=
\|A-B\|_F
}
$$

です。

これが、

> **mixed-canonical の左右環境が isometry なので、局所中心行列の誤差がそのままテンソル全体の誤差になる**

という意味です。


## 18. EYM を単一 Bond Truncation に適用する

bond rank を、

$$
r_2
\longrightarrow
k
$$

に制限したいとします。

局所行列では、

$$
\operatorname{rank}(B)\le k
$$

という rank 制約になります。

前節の isometry から、

$$
\min_{\operatorname{rank}(B)\le k}
\|X-X^{(B)}\|_F
=
\min_{\operatorname{rank}(B)\le k}
\|A-B\|_F.
$$

EYM 定理より、

$$
\min_{\operatorname{rank}(B)\le k}
\|A-B\|_F
=
\|A-A_k\|_F
=
\sqrt{
\sum_{\alpha=k+1}^{\rho}
\sigma_\alpha^2
}.
$$

したがって、

$$
\boxed{
\min_{\operatorname{rank}(B)\le k}
\|X-X^{(B)}\|_F
=
\sqrt{
\sum_{\alpha=k+1}^{\rho}
\sigma_\alpha^2
}
}
$$

です。

この最小値を達成する局所行列が、

$$
A_k
=
U_k\Sigma_kV_k^T
$$

です。


## 19. 実際に Bond Rank を $k$ へ縮めるときの注意

局所最適性を説明するときは、

$$
A_k
\in
\mathbb{R}^{(r_1n_2)\times r_2}
$$

を、元と同じ shape の近似中心行列として扱えます。

しかし、TT/MPS の **実際の bond dimension を $r_2\to k$ に縮める**には、$A_k$ をそのまま中心コアへ戻すだけではありません。

$$
A_k
=
U_k\Sigma_kV_k^T
$$

を、

$$
U_k
$$

と、

$$
\Sigma_kV_k^T
$$

に分けます。

第2コアを、

$$
\widetilde G_2^{[L]}
=
\operatorname{reshape}(U_k)
\in
\mathbb{R}^{r_1\times n_2\times k}
$$

とし、

$$
T_k
=
\Sigma_kV_k^T
\in
\mathbb{R}^{k\times r_2}
$$

を右 canonical コアへ吸収します。

$$
\widetilde G_3
=
T_k
G_3^{[R]}
\in
\mathbb{R}^{k\times n_3\times1}.
$$

これにより、

$$
\boxed{
r_2\rightarrow k
}
$$

という実際の bond rank 削減が実現します。

つまり、

1. $A_k$ は「同じ shape の最適 rank-$k$ 行列」として誤差解析に使う
2. $U_k$ と $\Sigma_kV_k^T$ に分けて隣接コアへ再配置することで、実際の TT bond を $k$ に縮める

という2段階を区別します。


## 20. 演習C — Mixed-Canonical 環境が誤差を保存することを確認する

### 目的

次の3つを数値で確認します。

### 1. 局所行列の誤差

$$
\|A-A_k\|_F.
$$

### 2. 同じ shape の近似中心行列として $A_k$ を戻した場合

$$
G_2^{(A_k)}
=
\operatorname{reshape}(A_k)
$$

として、

$$
X^{(A_k)}
=
G_1^{[L]}
G_2^{(A_k)}
G_3^{[R]}
$$

を作り、

$$
\boxed{
\|X-X^{(A_k)}\|_F
=
\|A-A_k\|_F
}
$$

を確認します。

### 3. 実際に bond rank を $k$ へ縮めた TT/MPS

$$
\widetilde G_2^{[L]}
=
\operatorname{reshape}(U_k),
$$

$$
\widetilde G_3
=
(\Sigma_kV_k^T)G_3^{[R]}
$$

から、

$$
\widetilde X
=
G_1^{[L]}
\widetilde G_2^{[L]}
\widetilde G_3
$$

を作ります。

そして、

$$
X^{(A_k)}
=
\widetilde X
$$

が丸め誤差内で一致することを確認します。

### TODO

1. `A_k` を構成する
2. `A_k` を元と同じ shape の中心コアへ戻して `X_from_Ak` を作る
3. 局所誤差と全体誤差を比較する
4. `U_k` を bond dimension $k$ の第2コアへ reshape する
5. `Sigma_k V_k^T` を `G3_right` へ吸収する
6. bond rank $k$ の `X_truncated` を再構成する
7. `X_from_Ak` と `X_truncated` が一致することを確認する

### 考えること

- `A_k` 自体の shape は元の `A` と同じなのに、なぜ rank は $k$ なのか
- 実際の TT core shape を小さくするには、なぜ再因子化と右コアへの吸収が必要なのか
- EYM の局所最適性が、なぜ単一 bond の TT/MPS 全体最適性へそのまま移るのか


In [6]:
# TODO C:
# mixed-canonical 環境の誤差保存と、
# 実際の bond rank k の TT/MPS を確認してください。
#
# 1. A_k を構成
A_k = U_k @ torch.diag(S_k) @ Vh_k
#
# 2. A_k を元と同じ shape の中心コアへ戻す
G2_from_Ak = A_k.reshape(r1, n2, r2)
X_from_Ak = reconstruct_tt3(G1_left, G2_from_Ak, G3_right)
#
# 3. ||A - A_k||_F と ||X - X_from_Ak||_F を比較
local_error = torch.linalg.vector_norm(A - A_k).item()
global_error = torch.linalg.vector_norm(X - X_from_Ak).item()
print("||A - A_k||_F         =", local_error)
print("||X - X_from_Ak||_F   =", global_error)
print("local vs global gap   =", abs(local_error - global_error))
#
# 4. U_k を bond dimension k の第2コアへ reshape
G2_truncated = U_k.reshape(r1, n2, k)
#
# 5. Sigma_k V_k^T を G3_right へ吸収
transfer_k = torch.diag(S_k) @ Vh_k
G3_truncated = torch.tensordot(transfer_k, G3_right, dims=([1], [0]))
#
# 6. bond rank k の X_truncated を再構成
X_truncated = reconstruct_tt3(G1_left, G2_truncated, G3_truncated)
#
# 7. X_from_Ak と X_truncated の一致を確認
bond_match_error = torch.linalg.vector_norm(X_from_Ak - X_truncated).item()
print("G2_truncated shape    :", tuple(G2_truncated.shape))
print("G3_truncated shape    :", tuple(G3_truncated.shape))
print("||X_from_Ak - X_k||_F =", bond_match_error)

||A - A_k||_F         = 0.25000000000000006
||X - X_from_Ak||_F   = 0.25
local vs global gap   = 5.551115123125783e-17
G2_truncated shape    : (2, 3, 2)
G3_truncated shape    : (2, 5, 1)
||X_from_Ak - X_k||_F = 1.436602425309578e-15


## 21. 作用素ノルム版との違い

EYM 定理には作用素ノルム版もあります。

作用素ノルムは、

$$
\|M\|_2
=
\max_{\|x\|_2=1}
\|Mx\|_2
$$

です。

rank-$k$ 近似について、

$$
\boxed{
\min_{\operatorname{rank}(B)\le k}
\|A-B\|_2
=
\sigma_{k+1}
}
$$

であり、truncated SVD がこの最小値を達成します。

Frobenius 版は、

$$
\|A-A_k\|_F
=
\sqrt{
\sigma_{k+1}^2
+
\cdots
+
\sigma_\rho^2
},
$$

作用素ノルム版は、

$$
\|A-A_k\|_2
=
\sigma_{k+1}.
$$

Notebook 09 の、

$$
\rho=3,
\qquad
k=2
$$

では捨てる特異値が1個だけなので、

$$
\|A-A_2\|_F
=
\|A-A_2\|_2
=
0.25.
$$

一般にはこの2つは異なります。

- Frobenius ノルム：捨てた成分全体の二乗和を測る
- 作用素ノルム：捨てた中で最大の1方向を測る

という違いがあります。


## 22. 今回のまとめ

### EYM の主張

$$
\boxed{
A_k
\in
\operatorname*{argmin}_{\operatorname{rank}(B)\le k}
\|A-B\|_F
}
$$

です。

候補 $B$ は、rank 条件を満たせば任意です。

### 一般 Frobenius 版

$$
\boxed{
\min_{\operatorname{rank}(B)\le k}
\|A-B\|_F^2
=
\sum_{\alpha=k+1}^{\rho}
\sigma_\alpha^2
}
$$

です。

### 証明の流れ

$$
\boxed{
\text{Ky Fan（引用）}
\rightarrow
\text{von Neumann}
\rightarrow
\|A-B\|_F^2
\ge
\sum_i(\sigma_i(A)-\sigma_i(B))^2
\rightarrow
\text{rank 制約}
\rightarrow
\text{EYM}
}
$$

です。

### TT/MPS の単一 bond では

mixed-canonical 環境が isometry なので、

$$
\boxed{
\|X-X^{(B)}\|_F
=
\|A-B\|_F
}
$$

です。

そのため EYM の局所最適性が、そのまま単一 bond truncation の全テンソル最適性へ移ります。

Notebook 09 の例では、

$$
\boxed{
\min \|X-\widetilde X\|_F
=
0.25
}
$$

です。

つまり Notebook 09 で得た `0.25` は単なる実測誤差ではなく、

> **第2–第3 bond を rank 2 に制限したときに達成できる最小 Frobenius 誤差**

です。

### 今回の実装確認

- 実験A：任意の rank-$2$ 候補は EYM 下界を破れない
- 実験B：von Neumann のトレース不等式
- 実験C：mixed-canonical 環境による局所誤差の保存

までを扱います。

rank の自動決定、tolerance、relative error、TT rounding、複数 bond の誤差評価には、この Notebook では進みません。
